# SQL/PostgreSQL — CADERNO PRÁTICO — SC TEC T8 (COLAB)**MATERIAL DIDÁTICO FICTÍCIO/SINTÉTICO — NÃO OFICIAL, NÃO AVALIAÇÃO FORMAL.** Espelha os oito checkpoints do Guia 05. A fonte do conteúdo é o conjunto de transcrições/artefatos SC TEC M1S08–M1S11, reconciliado sem atribuir ao professor exemplos criados aqui.**Como praticar:** 1. ENUNCIADO. 2. Entrada, saída e regra. 3. Previsão manual. 4. TENTATIVA (edite a célula). 5. Comparação com solução comentada. 6. Caso-limite e diagnóstico. 7. Recuperação ativa. No Colab use Shift+Enter em sessão temporária.**Fronteira:** o código executável deste Caderno é uma simulação **Python/SQLite ou dicionários Python**, não comprova conexão PostgreSQL ou MongoDB. Nunca utilizar dados reais, bancos da professora ou projetos avaliativos.

## Prepare o exemplo fictício

Execute uma única vez. Se reiniciar o kernel, reexecute a preparação antes das soluções. Não instalar servidores nem copiar credenciais.

In [ ]:
import sqlite3
cn=sqlite3.connect(":memory:")
cn.row_factory=sqlite3.Row
cn.executescript("""
PRAGMA foreign_keys=ON;
CREATE TABLE clientes(id INTEGER PRIMARY KEY,nome TEXT,estado TEXT);
CREATE TABLE produtos(id TEXT PRIMARY KEY,nome TEXT,preco NUMERIC);
CREATE TABLE pedidos(id INTEGER PRIMARY KEY,cliente_id INTEGER REFERENCES clientes(id),dia TEXT,status TEXT);
CREATE TABLE itens(pedido_id INTEGER REFERENCES pedidos(id),produto_id TEXT REFERENCES produtos(id),qtd INTEGER,preco NUMERIC,desconto NUMERIC,PRIMARY KEY(pedido_id,produto_id));
INSERT INTO clientes VALUES(1,'Lia','SP'),(2,'Raul','RJ'),(3,'Bia','MG');
INSERT INTO produtos VALUES('P01','Caderno',10),('P02','Caneta',20);
INSERT INTO pedidos VALUES(101,1,'2026-11-05','concluido'),(102,2,'2026-11-09','concluido');
INSERT INTO itens VALUES(101,'P01',2,10,0),(101,'P02',1,20,0),(102,'P01',3,5,10);
""")
print("Simulação SQLite na memória — NÃO é PostgreSQL conectado.")
assert cn.execute("SELECT COUNT(*) FROM clientes").fetchone()[0]==3

## Antes das 24 questões — diagnóstico e recuperação ativa**MATERIAL DIDÁTICO FICTÍCIO/SINTÉTICO; não constitui avaliação oficial.** A **Semana 08 começou por HTML, scraping e ETL**, depois PostgreSQL. Leia a [Ponte Semana 08](../../docs/ambientacao/ponte-semana-08-etl.md), execute a oficina local no Guia 05 e explique as etapas de extração → transformação → carga no SQLite. A base é fictícia e não acessa sites externos.Para cada checkpoint: (1) diga a pergunta de negócio; (2) escreva a previsão da **quantidade de linhas** e unidade; (3) responda sem abrir a solução; (4) execute no Python/SQLite; (5) compare com SQL PostgreSQL **somente no servidor de treino autorizado**. Use a [ambientação do PostgreSQL e pgAdmin](../../docs/ambientacao/primeiro-acesso-postgresql.md).**Oficina adicional de fixação (não numerada nem avaliada):** desenhe a tabela bruta com pedido, cliente, produto e quantidade; explique 1FN, 2FN e 3FN; separe clientes, produtos, pedidos e itens; identifique PK/FK; justifique por que `COUNT(*)` depois de JOIN pode superar `COUNT(DISTINCT pedido_id)`. Por fim, explique `WHERE` vs `HAVING` e por que novembro parcial não deve ser comparado diretamente com outubro completo.As células deste Caderno são **SQLite em memória**, não executam PostgreSQL instalado.

## 🧪 CHECKPOINT 1 — Questões 1–3

Q1. Qual diferença entre PostgreSQL, pgAdmin e SQL? Q2. O que significa localhost:5432 (sem presumir conexão)? Q3. Preveja 3 clientes, 2 pedidos e 3 linhas de item.

**Planejamento estruturado:** Entrada? Saída? Regra? Previsão de fronteira? Qual erro é provável?

In [ ]:
# TENTATIVA CP1
# Escreva sua previsão e explique antes de executar.
# Edite abaixo em seu ritmo. Não use dados reais nem cole respostas de prova.

### ✅ Solução comentada — confira somente após sua tentativa

**Q1:** PostgreSQL é o SGBD/servidor, pgAdmin é cliente e SQL é linguagem; Query Tool envia SQL. **Q2:** localhost refere-se ao computador do executor; 5432 é porta comum, não prova servidor ativo. **Q3:** o treino contém três clientes, dois pedidos e três linhas de itens. **Caso-limite:** abrir pgAdmin sem conexão não permite SELECT no banco.

A próxima célula demonstra uma verificação no conjunto fictício; não representa solução de avaliação institucional nem execução em PostgreSQL/MongoDB.

In [ ]:
print('SQLite local:',cn.execute('SELECT 1').fetchone()[0]);assert cn.execute('SELECT 1').fetchone()[0]==1

**Recuperação ativa do CP1:** explique com suas palavras o conceito, indique qual linguagem e **executor** seriam utilizados na prática real, mencione uma armadilha e proponha uma pergunta de negócio cuja resposta exige esse conteúdo.

## 🧪 CHECKPOINT 2 — Questões 4–6

Q4. Localize o cliente do estado MG. Q5. Qual a chave primária do cliente e a estrangeira do pedido? Q6. Por que SELECT * pode ser inconveniente?

**Planejamento estruturado:** Entrada? Saída? Regra? Previsão de fronteira? Qual erro é provável?

In [ ]:
# TENTATIVA CP2
# Escreva sua previsão e explique antes de executar.
# Edite abaixo em seu ritmo. Não use dados reais nem cole respostas de prova.

### ✅ Solução comentada — confira somente após sua tentativa

**Q4:** Bia está em MG. **Q5:** clientes.id é PK e pedidos.cliente_id referencia clientes.id como FK. **Q6:** SELECT * altera estrutura de saída quando novas colunas surgem e lê campos desnecessários; prefira nomear colunas. **Erro comum:** consultar schema errado.

A próxima célula demonstra uma verificação no conjunto fictício; não representa solução de avaliação institucional nem execução em PostgreSQL/MongoDB.

In [ ]:
rows=cn.execute("SELECT nome FROM clientes WHERE estado='SP'").fetchall();print([r['nome'] for r in rows]);assert rows[0]['nome']=='Lia'

**Recuperação ativa do CP2:** explique com suas palavras o conceito, indique qual linguagem e **executor** seriam utilizados na prática real, mencione uma armadilha e proponha uma pergunta de negócio cuja resposta exige esse conteúdo.

## 🧪 CHECKPOINT 3 — Questões 7–9

Q7. Compare COUNT(*) depois de JOIN pedido/itens e COUNT(DISTINCT pedido). Q8. Identifique cliente sem pedido. Q9. Onde um WHERE elimina ausência preservada pelo LEFT?

**Planejamento estruturado:** Entrada? Saída? Regra? Previsão de fronteira? Qual erro é provável?

In [ ]:
# TENTATIVA CP3
# Escreva sua previsão e explique antes de executar.
# Edite abaixo em seu ritmo. Não use dados reais nem cole respostas de prova.

### ✅ Solução comentada — confira somente após sua tentativa

**Q7:** JOIN pedidos×itens retorna três linhas; COUNT DISTINCT retorna dois pedidos. **Q8:** Bia não possui pedido (LEFT JOIN e IS NULL). **Q9:** WHERE p.status='concluido' após LEFT pode eliminar linhas nulas do lado opcional. **Caso-limite:** pedido com zero itens e filtros mal posicionados.

A próxima célula demonstra uma verificação no conjunto fictício; não representa solução de avaliação institucional nem execução em PostgreSQL/MongoDB.

In [ ]:
r=cn.execute('SELECT COUNT(*) a, COUNT(DISTINCT p.id) b FROM pedidos p JOIN itens i ON i.pedido_id=p.id').fetchone();assert (r['a'],r['b'])==(3,2);print(dict(r))

**Recuperação ativa do CP3:** explique com suas palavras o conceito, indique qual linguagem e **executor** seriam utilizados na prática real, mencione uma armadilha e proponha uma pergunta de negócio cuja resposta exige esse conteúdo.

## 🧪 CHECKPOINT 4 — Questões 10–12

Q10. Conte unidades e pedidos. Q11. Calcule manualmente 40,00 e 13,50. Q12. Diferencie WHERE, GROUP BY e HAVING.

**Planejamento estruturado:** Entrada? Saída? Regra? Previsão de fronteira? Qual erro é provável?

In [ ]:
# TENTATIVA CP4
# Escreva sua previsão e explique antes de executar.
# Edite abaixo em seu ritmo. Não use dados reais nem cole respostas de prova.

### ✅ Solução comentada — confira somente após sua tentativa

**Q10:** seis unidades e dois pedidos distintos. **Q11:** pedido 101: 2×10 +1×20=40; pedido 102: 3×5×0,9=13,50; soma 53,50. **Q12:** WHERE filtra linhas; GROUP BY agrupa; HAVING filtra resultados agregados. **Limite:** esse valor não é lucro nem recebimento.

A próxima célula demonstra uma verificação no conjunto fictício; não representa solução de avaliação institucional nem execução em PostgreSQL/MongoDB.

In [ ]:
r=cn.execute('SELECT COUNT(DISTINCT pedido_id) n,SUM(qtd) u,SUM(qtd*preco*(1-desconto/100.0)) v FROM itens').fetchone();assert (r['n'],r['u'],round(r['v'],2))==(2,6,53.5);print(dict(r))

**Recuperação ativa do CP4:** explique com suas palavras o conceito, indique qual linguagem e **executor** seriam utilizados na prática real, mencione uma armadilha e proponha uma pergunta de negócio cuja resposta exige esse conteúdo.

## 🧪 CHECKPOINT 5 — Questões 13–15

Q13. Pedido acima de 20 e cliente com soma acima de 20 são perguntas iguais? Q14. Uma média entre preços heterogêneos prova outlier? Q15. Por que a subconsulta em FROM necessita alias?

**Planejamento estruturado:** Entrada? Saída? Regra? Previsão de fronteira? Qual erro é provável?

In [ ]:
# TENTATIVA CP5
# Escreva sua previsão e explique antes de executar.
# Edite abaixo em seu ritmo. Não use dados reais nem cole respostas de prova.

### ✅ Solução comentada — confira somente após sua tentativa

**Q13:** pedido individual acima de limite não equivale a soma acumulada do cliente. **Q14:** preços de produtos heterogêneos tornam média geral inadequada como diagnóstico conclusivo de anomalia. **Q15:** subconsulta em FROM recebe alias para nomear a tabela derivada. **Caso-limite:** cliente com vários pedidos pequenos.

A próxima célula demonstra uma verificação no conjunto fictício; não representa solução de avaliação institucional nem execução em PostgreSQL/MongoDB.

In [ ]:
q='SELECT pedido_id FROM (SELECT pedido_id,SUM(qtd*preco*(1-desconto/100.0)) total FROM itens GROUP BY pedido_id) r WHERE total>20';rs=cn.execute(q).fetchall();assert [x[0] for x in rs]==[101];print('Pedidos >20:',[x[0] for x in rs])

**Recuperação ativa do CP5:** explique com suas palavras o conceito, indique qual linguagem e **executor** seriam utilizados na prática real, mencione uma armadilha e proponha uma pergunta de negócio cuja resposta exige esse conteúdo.

## 🧪 CHECKPOINT 6 — Questões 16–18

Q16. Monte resumo por pedido usando WITH. Q17. CTE é sempre mais rápida? Q18. O que ocorre se juntar novamente o resultado a uma tabela 1:N?

**Planejamento estruturado:** Entrada? Saída? Regra? Previsão de fronteira? Qual erro é provável?

In [ ]:
# TENTATIVA CP6
# Escreva sua previsão e explique antes de executar.
# Edite abaixo em seu ritmo. Não use dados reais nem cole respostas de prova.

### ✅ Solução comentada — confira somente após sua tentativa

**Q16:** WITH resumo AS (SELECT pedido_id, SUM(qtd) AS unidades FROM itens GROUP BY pedido_id) SELECT * FROM resumo; **Q17:** CTE não é inerentemente mais rápida; otimização depende do plano. **Q18:** novo JOIN 1:N sobre resumo pode multiplicar linhas. **Erro comum:** somar totais repetidos.

A próxima célula demonstra uma verificação no conjunto fictício; não representa solução de avaliação institucional nem execução em PostgreSQL/MongoDB.

In [ ]:
q='WITH r AS (SELECT pedido_id,SUM(qtd) unidades FROM itens GROUP BY pedido_id) SELECT COUNT(*) FROM r';assert cn.execute(q).fetchone()[0]==2;print('CTE: dois resumos de pedido')

**Recuperação ativa do CP6:** explique com suas palavras o conceito, indique qual linguagem e **executor** seriam utilizados na prática real, mencione uma armadilha e proponha uma pergunta de negócio cuja resposta exige esse conteúdo.

## 🧪 CHECKPOINT 7 — Questões 19–21

Q19. Preveja meses presentes no conjunto fictício. Q20. Novembro parcial prova queda frente a outubro completo? Q21. Como distinguir DATE_TRUNC de substr usada na simulação?

**Planejamento estruturado:** Entrada? Saída? Regra? Previsão de fronteira? Qual erro é provável?

In [ ]:
# TENTATIVA CP7
# Escreva sua previsão e explique antes de executar.
# Edite abaixo em seu ritmo. Não use dados reais nem cole respostas de prova.

### ✅ Solução comentada — confira somente após sua tentativa

**Q19:** há apenas novembro de 2026 no exemplo. **Q20:** novembro até dia 9 versus outubro completo é janela não comparável; analisar períodos equivalentes e sazonalidade. **Q21:** DATE_TRUNC('month', data)::date é sintaxe PostgreSQL; substr em SQLite produz texto, não date tipada. **Caso-limite:** ano e mês atravessam fronteiras.

A próxima célula demonstra uma verificação no conjunto fictício; não representa solução de avaliação institucional nem execução em PostgreSQL/MongoDB.

In [ ]:
r=cn.execute('SELECT substr(dia,1,7) mes,COUNT(*) n FROM pedidos GROUP BY substr(dia,1,7)').fetchall();print([tuple(x) for x in r]);assert len(r)==1

**Recuperação ativa do CP7:** explique com suas palavras o conceito, indique qual linguagem e **executor** seriam utilizados na prática real, mencione uma armadilha e proponha uma pergunta de negócio cuja resposta exige esse conteúdo.

## 🧪 CHECKPOINT 8 — Questões 22–24

Q22. Faça checklist antes de UPDATE no PostgreSQL. Q23. Verifique chaves estrangeiras no exemplo. Q24. Qual a diferença entre resultado de consulta e conclusão comercial válida?

**Planejamento estruturado:** Entrada? Saída? Regra? Previsão de fronteira? Qual erro é provável?

In [ ]:
# TENTATIVA CP8
# Escreva sua previsão e explique antes de executar.
# Edite abaixo em seu ritmo. Não use dados reais nem cole respostas de prova.

### ✅ Solução comentada — confira somente após sua tentativa

**Q22:** confirmar conexão/usuário/banco, WHERE, contagem prevista, transação e permissão antes de UPDATE. **Q23:** PRAGMA foreign_key_check sem linhas inconsistentes apenas comprova esta instância SQLite fictícia. **Q24:** SQL pode executar mas medir unidade errada ou excluir cancelados indevidamente; interpretação requer contexto. **Erro comum:** tratar saída de treino como prova institucional.

A próxima célula demonstra uma verificação no conjunto fictício; não representa solução de avaliação institucional nem execução em PostgreSQL/MongoDB.

In [ ]:
assert cn.execute('PRAGMA foreign_key_check').fetchall()==[];print('Sem violações de FK nos dados fictícios SQLite')

**Recuperação ativa do CP8:** explique com suas palavras o conceito, indique qual linguagem e **executor** seriam utilizados na prática real, mencione uma armadilha e proponha uma pergunta de negócio cuja resposta exige esse conteúdo.

## Revisão final e limites

Formule uma questão integradora que combine ao menos dois checkpoints. Registre uma tentativa original, explique o erro e a correção. **Aprovar as células deste Caderno não é prova de prática PostgreSQL/MongoDB, frequência, nota, submissão nem certificado institucional.** Os scripts dos bancos só devem ser usados em laboratório autorizado.